In [31]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.ui import Console
import asyncio
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
openai_model_client = OpenAIChatCompletionClient(model='gpt-4o',api_key=api_key)


In [32]:
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
import os
os.environ["SERPER_API_KEY"]=os.getenv("SERPER_API_KEY")
search_tool = GoogleSerperAPIWrapper()
def search_web(query: str)->str:
    return search_tool.run(query)


In [33]:
planning_agent = AssistantAgent(
    name = "PlanningAgent",
    model_client=openai_model_client,
    description= "An agent for planning tasks, this agent should be first to engage when given a new task.",
    system_message="""
    You are a planning agent
    your job is to break down complex tasks into smaller, managable tasks.
    Your team members are :
    WebSearchAgent: Searches information.
    DataAnalystAgent : Performs Calculations.

    You only plan and delegate tasks - you do not execute yourself.
    When assigning the task, use below format:
    1. <agent>:<task>

    After all the tasks are completed,summarize the findings and end with 'TERMINATE'
"""
)

web_search_agent = AssistantAgent(
    name = "WebSearchAgent",
    description="An Agent for searching the web for information.",
    model_client=openai_model_client,
    tools=[search_web],
    system_message='''
You are web search agent.
Your only tool is search_web_tool -use it to find the information you need.

You make only one search call at a time.

Once you have the result, you will never do the calculations or data analysis on them
'''
)



In [34]:
def percentage_change_tool(start:float,end:float) -> float:
    if start == 0:
        return 0
    return ((end-start)/start)*100

data_analysis_agent = AssistantAgent(
    name = "DataAnalysisAgent",
    description = "An agent for performing calculations and data analysis.",
    model_client=openai_model_client,
    tools=[percentage_change_tool],
    system_message='''
    You are a data anlayst agent.
    Given the task you have been assigned, you should analyze the data and provide results using the tools provided.

    If you have not seen the data, ask for it.

'''
)

In [35]:
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination
combined_termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=15)


In [36]:
from autogen_agentchat.teams import SelectorGroupChat
selector_prompt='''
{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
'''
selector_team = SelectorGroupChat(
    participants=[planning_agent,web_search_agent,data_analysis_agent],
    model_client=openai_model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True
)

task = "Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"


In [37]:
from autogen_agentchat.ui import Console

stream = selector_team.run_stream(task = task)
await Console(stream)

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To address this request, the tasks can be broken down as follows:

1. WebSearchAgent: Identify the Miami Heat player with the highest points in the 2006-2007 season.
2. WebSearchAgent: Find the total rebounds for that player during the 2007-2008 season.
3. WebSearchAgent: Find the total rebounds for that player during the 2008-2009 season.
4. DataAnalystAgent: Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons.

Let's proceed with these tasks. 

1. WebSearchAgent: Identify the Miami Heat player with the highest points in the 2006-2007 season.
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_GGXBoYuewoIYJHQBfNtTp3sS', arguments='{"query":"Miami

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 20, 6, 15, 21, 489091, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=162, completion_tokens=148), metadata={}, created_at=datetime.datetime(2025, 6, 20, 6, 15, 24, 717282, tzinfo=datetime.timezone.utc), content="To address this request, the tasks can be broken down as follows:\n\n1. WebSearchAgent: Identify the Miami Heat player with the highest points in the 2006-2007 season.\n2. WebSearchAgent: Find the total rebounds for that player during the 2007-2008 season.\n3. WebSearchAgent: Find the total rebounds for that player during the 2008-2009 season.\n4. DataAnalystAgent: Calculate the percentage cha

In [ ]:
openrouter_api_key ="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"
model_client = OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model = "deepseek/deepseek-r1-0528-qwen3-8b:free",
    api_key=openrouter_api_key,
    model_info={
        "family":"deepseek",
        "vision":True,
        "function_calling":True,
        "json_output":False
    }
    
)


ValueError: model_info is required when model name is not a valid OpenAI model

In [27]:
agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    description="You are a helpful agent."
)
stream = agent.run_stream(task = "What newtons laws of motion?")
await Console(stream)

---------- TextMessage (user) ----------
What newtons laws of motion?
---------- TextMessage (assistant) ----------
Newton's laws of motion are three fundamental principles that form the foundation of classical mechanics, formulated by Sir Isaac Newton in his work "Philosophiæ Naturalis Principia Mathematica" published in 1687. They describe the relationship between a body and the forces acting upon it, and the body's motion in response to those forces.

Here are the three laws:

1.  **Newton's First Law (Law of Inertia)**: An object at rest stays at rest, and an object in motion stays in motion with the same speed and in the same direction unless acted upon by an unbalanced external force.
    *   *Key Idea:* Objects don't change their state of motion on their own unless a force causes them to.

2.  **Newton's Second Law (F=ma)**: The acceleration of an object is directly proportional to the net force acting on it and inversely proportional to its mass. It can be summarized by the equ

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 20, 6, 4, 6, 918120, tzinfo=datetime.timezone.utc), content='What newtons laws of motion?', type='TextMessage'), TextMessage(source='assistant', models_usage=RequestUsage(prompt_tokens=48, completion_tokens=677), metadata={}, created_at=datetime.datetime(2025, 6, 20, 6, 4, 12, 2054, tzinfo=datetime.timezone.utc), content='Newton\'s laws of motion are three fundamental principles that form the foundation of classical mechanics, formulated by Sir Isaac Newton in his work "Philosophiæ Naturalis Principia Mathematica" published in 1687. They describe the relationship between a body and the forces acting upon it, and the body\'s motion in response to those forces.\n\nHere are the three laws:\n\n1.  **Newton\'s First Law (Law of Inertia)**: An object at rest stays at rest, and an object in motion stays in motion with the same speed and in the same direction unless acted upon 

In [42]:
from autogen_agentchat.messages import BaseAgentEvent,BaseChatMessage
from typing import Sequence,List
def selector_func(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    if messages[-1].source != planning_agent.name:
        return planning_agent.name
    return None
planning_agent = AssistantAgent(
    name = "PlanningAgent",
    model_client=openai_model_client,
    description= "An agent for planning tasks, this agent should be first to engage when given a new task.",
    system_message="""
    You are a planning agent
    your job is to break down complex tasks into smaller, managable tasks.
    Your team members are :
    WebSearchAgent: Searches information.
    DataAnalystAgent : Performs Calculations.

    You only plan and delegate tasks - you do not execute yourself.
    When assigning the task, use below format:
    1. <agent>:<task>

    After all the tasks are completed,summarize the findings and end with 'TERMINATE'
"""
)

web_search_agent = AssistantAgent(
    name = "WebSearchAgent",
    description="An Agent for searching the web for information.",
    model_client=openai_model_client,
    tools=[search_web],
    system_message='''
You are web search agent.
Your only tool is search_web_tool -use it to find the information you need.

You make only one search call at a time.

Once you have the result, you will never do the calculations or data analysis on them
'''
)

data_analysis_agent = AssistantAgent(
    name = "DataAnalysisAgent",
    description = "An agent for performing calculations and data analysis.",
    model_client=openai_model_client,
    tools=[percentage_change_tool],
    system_message='''
    You are a data anlayst agent.
    Given the task you have been assigned, you should analyze the data and provide results using the tools provided.

    If you have not seen the data, ask for it.

'''
)

selector_prompt='''
{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
'''
selector_team = SelectorGroupChat(
    participants=[planning_agent,web_search_agent,data_analysis_agent],
    model_client=openai_model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
    selector_func=selector_func
)

task = "Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"
stream = selector_team.run_stream(task = task)
await Console(stream)

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To address this request, we need to gather historical data about the Miami Heat players' performance during the specified seasons. Here are the tasks broken down:

1. WebSearchAgent: Search for the player with the highest points for Miami Heat in the 2006-2007 NBA season.
2. WebSearchAgent: Search for the total rebounds of that player in the 2007-2008 NBA season.
3. WebSearchAgent: Search for the total rebounds of the same player in the 2008-2009 NBA season.
4. DataAnalystAgent: Calculate the percentage change in total rebounds for the player between the 2007-2008 and 2008-2009 seasons.
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_885ouJOo0P2tnopdgFv6m5RK', arguments='{"query": "Mi

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 6, 20, 6, 26, 17, 93794, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=162, completion_tokens=143), metadata={}, created_at=datetime.datetime(2025, 6, 20, 6, 26, 19, 291593, tzinfo=datetime.timezone.utc), content="To address this request, we need to gather historical data about the Miami Heat players' performance during the specified seasons. Here are the tasks broken down:\n\n1. WebSearchAgent: Search for the player with the highest points for Miami Heat in the 2006-2007 NBA season.\n2. WebSearchAgent: Search for the total rebounds of that player in the 2007-2008 NBA season.\n3. WebSearchAgent: Search f

In [41]:
selector_team.reset()

<coroutine object BaseGroupChat.reset at 0x00000294A8606C50>